In [1]:
from google.colab import output
output.enable_custom_widget_manager()

## Simulatsiooni kood

See kood defineerib funktsiooni `simulate_lidar`, mis modelleerib drooni LiDAR mõõtmist.

Mida see teeb:
- Genereerib drooni trajektoori kahel vastassuunalisel lennuribal
- Rakendab sensorivead:
  - heading (IMU yaw viga)
  - timing offset (GNSS–LiDAR ajanihe)
  - lever arm (sensori offset drooni suhtes)
- Simuleerib LiDAR kiiri juhusliku nurga all (±30°)
- Arvutab, kuhu iga kiir maapinnal tabab
- Kontrollib, kas kiir tabab objekti enne maad
- Tagastab kõik punktid 3D koordinaatides

See on kogu simulatsiooni “tuum”, mida hiljem visualiseerime.

In [2]:
import numpy as np

def simulate_lidar(
    heading_error_deg=0,
    timing_offset_ms=0,
    lever_arm_x_m=0
):
    speed = 5.0
    altitude = 40.0
    lidar_rate = 100.0
    duration = 20.0
    n_points = int(duration * lidar_rate)

    scan_angle_max = np.deg2rad(30)

    # objektid
    objects_xy = np.array([
        [20, 2],
        [40, -3],
        [60, 5],
        [80, 0],
        [50, 8]
    ])
    object_height = 2.0

    def generate_strip(y_const, forward=True):
        t = np.linspace(0, duration, n_points)

        if forward:
            x = speed * t
            heading = 0.0
        else:
            x = 100 - speed * t
            heading = np.pi

        y = np.full_like(x, y_const)
        h = np.full_like(x, heading)

        return t, x, y, h

    # kaks riba (vastassuunas!)
    t1, x1, y1, h1 = generate_strip(0, True)
    t2, x2, y2, h2 = generate_strip(10, False)

    t = np.concatenate([t1, t2])
    x = np.concatenate([x1, x2])
    y = np.concatenate([y1, y2])
    heading = np.concatenate([h1, h2])

    # --- timing offset ---
    t_offset = t + timing_offset_ms / 1000.0
    x = np.interp(t_offset, t, x)
    y = np.interp(t_offset, t, y)
    heading = np.interp(t_offset, t, heading)

    # --- heading error ---
    heading = heading + np.deg2rad(heading_error_deg)

    # --- lever arm (body → world) ---
    dx = lever_arm_x_m * np.cos(heading)
    dy = lever_arm_x_m * np.sin(heading)

    x_lidar = x + dx
    y_lidar = y + dy
    z_lidar = altitude

    # --- scan ---
    scan_angles = np.random.uniform(-scan_angle_max, scan_angle_max, size=len(x))

    pts = []

    for i in range(len(x)):
        theta = scan_angles[i]

        ground_dist = altitude * np.tan(theta)

        # risti lennusuunaga
        dir_angle = heading[i] + np.pi / 2

        gx = x_lidar[i] + ground_dist * np.cos(dir_angle)
        gy = y_lidar[i] + ground_dist * np.sin(dir_angle)

        gz = 0.0

        # objekti tabamus
        for obj in objects_xy:
            if np.hypot(gx - obj[0], gy - obj[1]) < 1.0:
                gz = object_height

        pts.append([gx, gy, gz])

    return np.array(pts), objects_xy

## Visualiseerimine ja interaktiivsus

See kood joonistab simulatsiooni tulemuse ning lisab interaktiivsed slaiderid.

Mida see teeb:
- Jagab punktid kaheks lennuribaks (sinine ja punane)
- Kuvab objektide tegelikud asukohad (rohelised ristid)
- Kasutab Plotly't graafiku joonistamiseks
- Lisab 3 slaiderit:
  - heading_error_deg
  - timing_offset_ms
  - lever_arm_x_m
- Uuendab graafikut reaalajas, kui slaidereid liigutada

Oluline detail:
Telgede skaala on lukustatud (scaleanchor='y'),
et geomeetrilised moonutused oleksid visuaalselt õiged.

In [3]:
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider
from IPython.display import display

pts, objects = simulate_lidar(0, 0, 0)
pts0 = np.array(pts)

n = len(pts0) // 2

fig = go.FigureWidget([
    go.Scatter(
        x=pts0[:n, 0], y=pts0[:n, 1],
        mode='markers',
        marker=dict(color='blue', size=3),
        name='Riba 1'
    ),

    go.Scatter(
        x=pts0[n:, 0], y=pts0[n:, 1],
        mode='markers',
        marker=dict(color='red', size=3),
        name='Riba 2'
    ),

    go.Scatter(
        x=objects[:, 0], y=objects[:, 1],
        mode='markers',
        marker=dict(color='green', size=13, symbol='x'),
        name='Objektid'
    ),
])

fig.update_layout(
    width=750,
    height=650,
    xaxis=dict(title="x [m]", scaleanchor="y"),
    yaxis=dict(title="y [m]"),
    title="LiDAR mõõtmiste simulaator",
)

def update(heading_error_deg, timing_offset_ms, lever_arm_x_m):

    pts, _ = simulate_lidar(
        heading_error_deg,
        timing_offset_ms,
        lever_arm_x_m
    )

    n = len(pts) // 2
    strip1 = pts[:n]
    strip2 = pts[n:]

    with fig.batch_update():
        fig.data[0].x = strip1[:,0]
        fig.data[0].y = strip1[:,1]

        fig.data[1].x = strip2[:,0]
        fig.data[1].y = strip2[:,1]

        fig.layout.title.text = (
            f"heading={heading_error_deg:+.2f}° | "
            f"timing={timing_offset_ms:+.0f} ms | "
            f"lever={lever_arm_x_m:+.2f} m"
        )
display(fig)

interact(
    update,
    heading_error_deg=FloatSlider(min=-5, max=5, step=0.1, value=0),
    timing_offset_ms=FloatSlider(min=-200, max=200, step=10, value=0),
    lever_arm_x_m=FloatSlider(min=-0.5, max=0.5, step=0.01, value=0)
)

FigureWidget({
    'data': [{'marker': {'color': 'blue', 'size': 3},
              'mode': 'markers',
              'name': 'Riba 1',
              'type': 'scatter',
              'uid': '262054c0-1f77-411c-a07c-620210e01363',
              'x': array([ 5.19697722e-16,  5.00250125e-02,  1.00050025e-01, ...,  9.98999500e+01,
                           9.99499750e+01, -2.79047044e-15]),
              'y': array([  8.48730789,  12.68507513, -17.96853084, ...,   0.49817151,
                           17.63183145,  -5.19061354])},
             {'marker': {'color': 'red', 'size': 3},
              'mode': 'markers',
              'name': 'Riba 2',
              'type': 'scatter',
              'uid': 'd66fde6d-65b5-4ab5-82d9-3275d51798f1',
              'x': array([ 7.94992137e-16,  5.00250125e-02,  1.00050025e-01, ...,  9.98999500e+01,
                           9.99499750e+01, -2.08348626e-15]),
              'y': array([12.98320688, -1.58925958,  5.91371097, ..., 21.36376569, -1.50852   

interactive(children=(FloatSlider(value=0.0, description='heading_error_deg', max=5.0, min=-5.0), FloatSlider(…

<function __main__.update(heading_error_deg, timing_offset_ms, lever_arm_x_m)>

## Simulatsiooni analüüs
1. Kui heading kasvab ajas 20 sekundi jooksul 0° kuni 1°, siis lennu alguses peaaegu pole viga (0° juures) ning lennu lõpus on maksimaalne viga (1° juures). See tähendab, et headingu suurenedes kasvab ka nihe.
Konstantset (heading = 1°) viga on lihtne parandada ehk kalibreerida, sest nihe jääb kogu stripi vältel samaks. Kui aga tegu oleks driftiga (0.05°/s), siis nihe kasvaks ajas ning viga suureneks. Samuti selline strip jääb "venima" ning on keerulisem pidevate muutustega korrigeerida.
2. Konstantne viga (näiteks heading = 1°) on sama terve lennu jooksul ning ei sõltu ka distantsist. Konstantne viga sõltub seljuhul geomeetriast (kõrgus + scan angle).
Pika (9000 m) lennu puhul domineerivad aja-kogunevad vead. Aja-kogunev viga (drift) skaleerub ehk kasvab ajas. Lennu alguses on viga minimaalne ning lennu lõpus maksimaalne. Samuti kasvab ka punktide ruumiline nihe distantsiga.
3. IMU täpsus loeb rohkem, sest IMU viga keerab kogu mõõtmise suunda, samal ajal kui GNSS viga nihutab punkti.
Antud on kõrgus = 40 m; scan = 30°; GNSS viga = 1 cm ning IMU viga = 0.1°.
Esimese asjana tuleks leida kaugus servast (r): r = 40 m * tan(30°) ≈ 23 m
Teisena tuleks teisendada IMU vea nurk radiaanideks: 0.1° = 0.001745 rad
Kolmandaks saame leida positsioonivea valemiga: Δ = r * θ = 23 * 0.001745 ≈ 0.04 m ehk 4 cm. Siit saab näha, et IMU viga (4 cm) on suurem kui GNSS viga (1 cm), sest IMU viga skaleerub kaugusega. Mida kaugem punkt, seda suurem viga.